# Datathon Passos Mágicos — Limpeza da Base PEDE (2022, 2023, 2024)

Este notebook faz a **limpeza e padronização** da planilha `BASE_DE_DADOS_PEDE_2024_-_DATATHON.xlsx`,
que contém três abas (`PEDE2022`, `PEDE2023`, `PEDE2024`) com estrutura e nomenclatura de colunas
diferentes entre si.

**O que este notebook faz:**
1. Carrega as três abas
2. Remove colunas 100% vazias e colunas duplicadas
3. Corrige inconsistências de digitação/acentuação (`Gênero`, `Pedra`)
4. Corrige um bug de importação do Excel na aba 2023 (`Idade` / `Data de Nasc`)
5. Corrige a coluna `Fase` da aba 2024, que veio como código de turma em vez do número da fase
6. Padroniza nomes de colunas entre os três anos
7. Consolida tudo em uma tabela única em formato longo (1 linha = 1 aluno em 1 ano)
8. Valida os dados limpos (duplicidade de RA, faixa de valores dos indicadores)
9. Exporta os resultados (`.xlsx` e `.csv`)

> Ajuste o caminho do arquivo na célula abaixo (`RAW_PATH`) antes de rodar.

> **Rodando no Google Colab?** A célula abaixo clona o repositório automaticamente e ajusta o diretório de trabalho — não precisa fazer nada manualmente, só executar as células em ordem, de cima para baixo.

In [ ]:
import os

# Detecta se está rodando no Google Colab. Se estiver, clona o repositório (ou entra
# nele, se já tiver sido clonado nesta sessão) e muda o diretório de trabalho para
# dentro de notebooks/ -- assim todo o resto do notebook (que usa caminhos relativos
# à pasta notebooks/) funciona igual, esteja você no Colab, no Jupyter local ou em
# qualquer outro ambiente.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/juliamchaves/dathatonfiap.git"
    REPO_DIR = "/content/dathatonfiap"

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        # Repositório já clonado nesta sessão -- só atualiza (não recloná do zero).
        get_ipython().system(f"cd {REPO_DIR} && git pull")

    os.chdir(f"{REPO_DIR}/notebooks")
    print("Rodando no Colab -- diretório de trabalho:", os.getcwd())
else:
    print("Rodando fora do Colab -- usando os arquivos locais do repositório clonado.")


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Este notebook mora em notebooks/, então a raiz do repositório é o diretório pai.
# Adicionamos ela ao sys.path para poder importar as funções de limpeza de src/limpeza.py
# em vez de redefini-las aqui -- assim a lógica de limpeza fica documentada em um único
# lugar (usada também, se necessário, por outros scripts do projeto).
RAIZ_PROJETO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from src.limpeza import (
    PEDRA_FIX,
    COLUNAS_COMUNS,
    drop_fully_empty_columns,
    drop_duplicated_content_columns,
    strip_strings,
    fix_accent_variants,
    clean_2022,
    clean_2023,
    clean_2024,
    select_common,
)

pd.set_option("display.max_columns", 60)

RAW_PATH = RAIZ_PROJETO / "data" / "raw" / "BASE_DE_DADOS_PEDE_2024_-_DATATHON.xlsx"  # ajuste se necessário

## 1. Leitura dos dados brutos

In [ ]:
df_2022_raw = pd.read_excel(RAW_PATH, sheet_name="PEDE2022")
df_2023_raw = pd.read_excel(RAW_PATH, sheet_name="PEDE2023")
df_2024_raw = pd.read_excel(RAW_PATH, sheet_name="PEDE2024")

print("PEDE2022:", df_2022_raw.shape)
print("PEDE2023:", df_2023_raw.shape)
print("PEDE2024:", df_2024_raw.shape)

### Diagnóstico rápido dos problemas encontrados

Antes de limpar, vale registrar o que foi identificado ao inspecionar a base (isso justifica cada
etapa de limpeza feita mais abaixo):

- **Colunas 100% vazias** em 2023 e 2024 (`Cg`, `Cf`, `Ct`, `Rec Av1`...`Rec Av4`, `Indicado`,
  `Atingiu PV`, `Destaque IEG/IDA/IPV`, `Rec Psicologia`) — existem na planilha mas nunca foram
  preenchidas nesses anos.
- **Colunas duplicadas** (`Destaque IPV.1` em 2023, `Ativo/ Inativo.1` em 2024) com o mesmo
  conteúdo da coluna original.
- **`Gênero`** usa vocabulário diferente por ano: `Menina`/`Menino` (2022) vs. `Feminino`/`Masculino`
  (2023 e 2024).
- **`Pedra`** (categoria de desempenho) aparece grafada ora como `Ágata`, ora como `Agata`
  (sem acento) dentro da mesma coluna.
- **Bug de importação do Excel em 2023**: a coluna `Idade` tem parte dos valores convertidos para
  datas (`datetime(1900, 1, 8)` em vez do número `8`), e `Data de Nasc` mistura texto
  (`'6/17/2015'`) com objetos `datetime` já convertidos.
- **`Fase` em 2024** não contém o número da fase, e sim o código da turma (`'1A'`, `'8B'`, `'ALFA'`),
  diferente do que ocorre em 2022 (número puro) e 2023 (`'FASE 1'`...`'FASE 8'`, `'ALFA'`).
- **38 alunos em 2024** aparecem com `Fase = '9'` e `Pedra 2024 = 'INCLUIR'` — não é uma fase real,
  e sim um cadastro pendente de definição de turma.
- Nomes de colunas equivalentes variam entre anos (`Matem`/`Mat`, `Portug`/`Por`, `Inglês`/`Ing`,
  `INDE 22`/`INDE 2023`/`INDE 2024`, `Fase ideal`/`Fase Ideal`, `Defas`/`Defasagem` etc.).

## 2. Funções auxiliares de limpeza

As funções abaixo (`drop_fully_empty_columns`, `drop_duplicated_content_columns`,
`strip_strings`, `fix_accent_variants`, `clean_2022`, `clean_2023`, `clean_2024`,
`select_common`) **não estão definidas neste notebook** — foram importadas de
`src/limpeza.py` na célula de setup, para que a lógica de limpeza fique num único
lugar reutilizável, em vez de duplicada entre notebook e scripts. O código abaixo só
mostra a assinatura/uso; para ver a implementação completa, abra `src/limpeza.py`.

In [ ]:
# Já importadas de src/limpeza.py (ver célula de setup) -- nada a definir aqui.
print("Funções de limpeza disponíveis:", [
    drop_fully_empty_columns.__name__,
    drop_duplicated_content_columns.__name__,
    strip_strings.__name__,
    fix_accent_variants.__name__,
])

## 3. Limpeza — PEDE 2022

Colunas específicas: `Ano nasc`, `Idade 22`, `INDE 22`, `Pedra 22`, `Matem`, `Portug`, `Inglês`,
`Fase ideal`, `Defas`. `Fase` já vem como número (0 a 7).

In [ ]:
# clean_2022 já foi importada de src/limpeza.py -- só chamamos aqui.
df_2022, dropped_2022 = clean_2022(df_2022_raw)
print("Colunas removidas em 2022:", dropped_2022)
print("Shape final 2022:", df_2022.shape)
df_2022.head(3)

## 4. Limpeza — PEDE 2023

Além do padrão de renomeação, esta aba tem dois problemas específicos:

- **Colunas totalmente vazias / duplicadas** (ver diagnóstico acima)
- **Bug de importação do Excel**: valores pequenos de `Idade` viraram objetos `datetime`
  (`datetime(1900, 1, D)`, onde `D` é a idade real) porque a célula estava formatada como data.
  A correção extrai o dia (`.day`) desses casos e mantém os demais como número.
- **`Data de Nasc` mista** texto (`'6/17/2015'`) com `datetime` — padronizada via `pd.to_datetime`.
- **`Fase` em texto** (`'ALFA'`, `'FASE 1'`...`'FASE 8'`) — extraído o número em `Fase_Num`
  (`ALFA` → `0`).

In [ ]:
# clean_2023 já foi importada de src/limpeza.py -- só chamamos aqui.
df_2023, dropped_2023 = clean_2023(df_2023_raw)
print("Colunas removidas em 2023:", dropped_2023)
print("Shape final 2023:", df_2023.shape)
print("Idade após correção (amostra):", sorted(df_2023["Idade"].dropna().unique())[:15])
df_2023.head(3)

## 5. Limpeza — PEDE 2024

Problema específico desta aba: a coluna `Fase` **não contém o número da fase**, e sim o
**código de turma** (`'1A'`, `'8B'`, `'ALFA'`...). O número da fase é extraído do primeiro dígito
do código (`ALFA` → `0`).

Também há 38 alunos com `Fase == '9'` e `Pedra 2024 == 'INCLUIR'`, que representam **cadastros
pendentes** (ainda sem turma/fase definida) e não uma fase real — eles são sinalizados na coluna
`Pendente_Inclusao` em vez de descartados, para não perder o registro do aluno.

In [ ]:
# clean_2024 já foi importada de src/limpeza.py -- só chamamos aqui.
df_2024, dropped_2024 = clean_2024(df_2024_raw)
print("Colunas removidas em 2024:", dropped_2024)
print("Shape final 2024:", df_2024.shape)
print("Alunos pendentes de inclusão:", df_2024["Pendente_Inclusao"].sum())
df_2024.head(3)

## 6. Validação dos dados limpos

In [ ]:
# RA não pode se repetir dentro do mesmo ano
for nome, df in [("2022", df_2022), ("2023", df_2023), ("2024", df_2024)]:
    dup = df["RA"].duplicated().sum()
    status = "OK" if dup == 0 else f"ALERTA: {dup} duplicados"
    print(f"RA duplicado em {nome}: {status}")

In [ ]:
# Indicadores (IAA, IEG, IPS, IPP, IDA, IPV, IAN, INDE) devem estar na faixa 0-10
INDICADORES = ["IAA", "IEG", "IPS", "IPP", "IDA", "IPV", "IAN", "INDE"]

for nome, df in [("2022", df_2022), ("2023", df_2023), ("2024", df_2024)]:
    for c in INDICADORES:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            fora = s[(s < 0) | (s > 10.5)]
            if len(fora):
                print(f"[ALERTA] {nome} / {c}: {len(fora)} valores fora da faixa esperada (0-10)")

print("Validação de faixa concluída (sem mensagens acima = tudo dentro do esperado).")

## 7. Consolidação em formato longo

Uma linha = **1 aluno em 1 ano**, com as colunas que existem de forma equivalente nos três anos.
Isso facilita comparações ao longo do tempo (2022 → 2023 → 2024), que é a base para responder às
perguntas de evolução de IAN, IDA, IEG etc. do desafio.

In [ ]:
# COLUNAS_COMUNS e select_common já foram importadas de src/limpeza.py.
df_long = pd.concat(
    [select_common(df_2022), select_common(df_2023), select_common(df_2024)],
    ignore_index=True,
)

print("Shape do consolidado:", df_long.shape)
df_long.groupby("Ano")["RA"].nunique()

In [ ]:
df_long.head(10)

## 8. Exportação

Gera:
- `PEDE_limpo.xlsx` — uma aba por ano já limpa, mais a aba `Consolidado_Longo`
- `PEDE_consolidado_longo.csv` — a versão consolidada em CSV, pronta para as análises e para o
  modelo preditivo

In [ ]:
SAIDA = RAIZ_PROJETO / "data" / "raw"
SAIDA.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(SAIDA / "PEDE_limpo.xlsx") as writer:
    df_2022.to_excel(writer, sheet_name="PEDE2022_limpo", index=False)
    df_2023.to_excel(writer, sheet_name="PEDE2023_limpo", index=False)
    df_2024.to_excel(writer, sheet_name="PEDE2024_limpo", index=False)
    df_long.to_excel(writer, sheet_name="Consolidado_Longo", index=False)

df_long.to_csv(SAIDA / "PEDE_consolidado_longo.csv", index=False)

print("Arquivos exportados em:", SAIDA)

## 9. Resumo das decisões de limpeza

| Problema encontrado | Decisão tomada |
|---|---|
| Colunas 100% vazias em 2023/2024 | Removidas (sem qualquer informação) |
| Colunas duplicadas (`.1`) | Removidas quando o conteúdo era idêntico à original |
| `Gênero` com vocabulário diferente por ano | Padronizado para `Feminino`/`Masculino` |
| `Pedra` com/sem acento (`Agata`/`Ágata`) | Padronizado para `Ágata` |
| `Idade` 2023 corrompida como data pelo Excel | Extraído o dia (`.day`) dos valores afetados |
| `Data de Nasc` 2023 mista texto/datetime | Padronizada via `pd.to_datetime` |
| `Fase` 2023 em texto (`FASE 1`...`ALFA`) | Extraído número em `Fase_Num` (`ALFA`→`0`) |
| `Fase` 2024 = código de turma, não a fase | Extraído número em `Fase_Num` a partir do código |
| 38 alunos com `Fase='9'`/`Pedra='INCLUIR'` em 2024 | Sinalizados em `Pendente_Inclusao`, não descartados |
| Nomes de colunas divergentes entre anos | Padronizados (`Matem`/`Mat`→`Matematica` etc.) |
| Espaços extras em campos de texto | Removidos em todas as colunas de texto |

**Observação:** `Instituicao_Ensino` tem semântica diferente entre anos — em 2022 é o nome da rede/escola,
em 2023/2024 é o tipo (`Pública`/`Privada...`). Em 2024 o nome da escola em si está na coluna
`Nome_Escola`. Isso foi mantido como está (não é um erro de digitação, é uma mudança de metodologia
de coleta entre os anos), mas é importante ter isso em mente ao cruzar essa coluna entre os três anos.